<a href="https://colab.research.google.com/github/sanyamsh7/Agentic-RAG-Pipeline-From-Scratch/blob/main/Module4_RAG_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MODULE 4 - VECTOR DATABASES & RETRIEVAL - Storing and finding embeddings

## Core Concepts:

1. Why Vector Databases?

    - The library analogy: Index vs. checking every book
    - Speed comparison: Seconds → Milliseconds
    - Real-world scale: Handling millions of vectors


2. ChromaDB Fundamentals:

    - Collections, Documents, Embeddings, Metadata
    - Distance metrics (L2, Cosine)
    - HNSW indexing (how it's fast!)


3. LangChain Integration:

    - The easy way to use vector databases
    - Automatic embedding creation
    - Clean Document interface

4. Retrieval Strategies:
    - Similarity Search -> Most relevant results
    - MMR -> Diverse results ( avoids redundancy )
    - Score Threshold -> Quality control

5. Hybrid Search:
    - Vector similarity + Metadata Filtering = power move!
    - Example: "Find AI papers from 2023 only"

6. Complete Retrieval System:
    - Build a production-ready retriever class.
    - Configurable parameters
    - Ready for RAG!

## Architecture to Build
```
User Query
    |
Embed Query (384D vector)
    |
Vector Database Search (ChromaDB)
    |
[HNSW Index finds nearest neighbors]
    |
Top-K Most Similar Chunks
```
LEARNING OBJECTIVES:
1. Understand what vector databases are and why we need them
2. Learn how ChromaDB stores and searches embeddings
3. Explore indexing strategies (HNSW, IVF, etc.)
4. Master the retrieval pipeline (query → search → results)
5. Use metadata filtering for better results
6. Build a complete retrieval system


In [ ]:
# installing dependencies
!pip install chromadb sentence-transformers langchain langchain-community langchain-chroma -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 91.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 70.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 81.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71

In [ ]:
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document
import numpy as np
from typing import List, Dict
import time

## Understanding ChromaDb basics

In [ ]:
client = chromadb.Client(Settings(
    anonymized_telemetry = False, # disablign telemetry
    allow_reset = True # allows resetting the database
))

1. CLIENT: The database connection
2. COLLECTION: A group of embeddings (like a table in SQL)
3. DOCUMENTS: The text you're storing
4. EMBEDDINGS: The vectors (computed automatically or provided)
5. METADATA: Extra info about each document (source, date, etc.)
6. IDS: Unique identifiers for each item

In [ ]:
# creating our first collection
# delete collection if it exists for clean slate
try:
  client.delete_collection("demo_collection")
except:
  pass

# create a new collection
collection = client.create_collection(
    name = "demo_collection",
    metadata = {"description": "My first vector database"}
)

print("Collection 'demo_collection' created!")
print(f"Collection count: {collection.count()} items")

Collection 'demo_collection' created!
Collection count: 0 items


In [1]:
# add some documents
documents = [
    "The quick brown fox jumps over the lazy dog",
    "Machine learning is a subset of artificial intelligence",
    "Python is a popular programming language for data science",
    "The Eiffel Tower is located in Paris, France",
    "Photosynthesis is the process by which plants make food",
]

# load embedding model
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# creating embeddings
embeddings = embedding_model.encode(documents).tolist()

# add a collection
collection.add(
    documents = documents,
    embeddings = embeddings,
    ids=[f"doc_{i}" for i in range(len(documents))],
    metadatas = [{"source": "demo", "index": i} for i in range(len(documents))]
)

print(f"Collection count: {collection.count()} items added to collection")

In [ ]:
# querying the vector database
query = "What is AI and machine learning?"
print(query)

# creating query embeddings
query_embedding = embedding_model.encode([query]).tolist()

# search
print("Searching ...")
results = collection.query(
    query_embeddings = query_embedding,
    n_results = 3 # to get top 3 results
)

print(f"found {len(results['documents'])} results")

What is AI and machine learning?
Searching ...
found 1 results


In [ ]:
# diplay results
for i, (doc, distance, metadata) in enumerate(zip(
    results['documents'][0],
    results['distances'][0],
    results['metadatas'][0]), 1):
  # chromadb returns L2 distance , lower is better
  # convert to similarity score ( inverse )
  similarity = 1 / (1 + distance)
  print(f"\n{i}. Similarity: {similarity:.4f} | Distance: {distance:.4f}")
  print(f"   Document: {doc}")
  print(f"   Metadata: {metadata}")


1. Similarity: 0.6806 | Distance: 0.4692
   Document: Machine learning is a subset of artificial intelligence
   Metadata: {'source': 'demo', 'index': 1}

2. Similarity: 0.4073 | Distance: 1.4553
   Document: Python is a popular programming language for data science
   Metadata: {'index': 2, 'source': 'demo'}

3. Similarity: 0.3645 | Distance: 1.7438
   Document: Photosynthesis is the process by which plants make food
   Metadata: {'source': 'demo', 'index': 4}


##Understanding distance metrics -

Chromadb uses distance ( not similarity ):
- Lower distance = more similar
- Higher distance = less similar

COMMON METRICS:

1. L2 (Euclidean) Distance - DEFAULT in ChromaDB
   - Like measuring straight-line distance
   - Range: 0 to ∞
   - 0 = identical, higher = more different
   
2. Cosine Distance
   - Measures angle between vectors
   - Range: 0 to 2
   - 0 = identical, 2 = opposite
   
3. Inner Product
   - Dot product of vectors
   - Range: -∞ to ∞
   - Higher = more similar

FOR MOST RAG CASES: L2 or Cosine work equally well!

COMVERSIONS:

L2 distance → Similarity score:

  similarity = 1 / (1 + distance)

Cosine distance → Cosine similarity:

  cosine_similarity = 1 - cosine_distance

## Langchain Integration - High level APIs

Chomadb with Langchain - much easier to use!

Benefits:-
1.  Automatic embedding creation
2. Document management built-in
3. Metadata handling simplified
4. Integrates with LangChain ecosystem

In [2]:
# creating langchain compatible embeddings
lc_embeddings = HuggingFaceEmbeddings(
    model_name = "all-MiniLM-L6-v2"
)

# create documents with metadata
lc_documents = [
    Document(
        page_content = "The solar system has eight planets orbiting the sun",
        metadata = {"topic":"astronomy", "difficulty": "easy"}
    ),
    Document(
        page_content="Neural networks use backpropagation for training",
        metadata={"topic": "AI", "difficulty": "hard"}
    ),
    Document(
        page_content="Water boils at 100 degrees Celsius at sea level",
        metadata={"topic": "physics", "difficulty": "easy"}
    ),
    Document(
        page_content="The mitochondria is the powerhouse of the cell",
        metadata={"topic": "biology", "difficulty": "medium"}
    ),
]

# chromadb vector store
# Pass the existing client instance to avoid conflicts
vectorstore = Chroma.from_documents(
    documents = lc_documents,
    embedding = lc_embeddings,
    collection_name = "langchain_demo",
    client = client # Explicitly pass the client
)

print(f"Documents stored: {vectorstore._collection.count()}")

In [ ]:
# simple retrieval with Langchain
query = "How do brain-inspired computing systems learn?"

# retrieve similar documents
print("Retrieving relevant documents...")
retrieved_docs = vectorstore.similarity_search(query, k = 2)

print(f"Found {len(retrieved_docs)} documents")

Retrieving relevant documents...
Found 2 documents


In [ ]:
# results
for i, doc in enumerate(retrieved_docs,1):
  print(f"\n{i}. Topic: {doc.metadata['topic']} | Difficulty: {doc.metadata['difficulty']}")
  print(f"   Content: {doc.page_content}")


1. Topic: AI | Difficulty: hard
   Content: Neural networks use backpropagation for training

2. Topic: biology | Difficulty: medium
   Content: The mitochondria is the powerhouse of the cell


In [ ]:
# retrieval with similarity scores
query = "tell me about space"
print(query)

results_with_scores = vectorstore.similarity_search_with_score(query, k=3)

for i, (doc, score) in enumerate(results_with_scores, 1):
  # converting distance to similarity
  similarity = 1/(1+score)
  bar = "█" * int(similarity * 30)
  print(f"\n{i}. Score: {score:.4f} | Similarity: {similarity:.4f}")
  print(f"   {bar}")
  print(f"   Topic: {doc.metadata['topic']}")
  print(f"   Content: {doc.page_content}")

tell me about space

1. Score: 1.4329 | Similarity: 0.4110
   ████████████
   Topic: astronomy
   Content: The solar system has eight planets orbiting the sun

2. Score: 1.7620 | Similarity: 0.3621
   ██████████
   Topic: biology
   Content: The mitochondria is the powerhouse of the cell

3. Score: 1.8422 | Similarity: 0.3518
   ██████████
   Topic: AI
   Content: Neural networks use backpropagation for training


## Metadata Filtering - Hybrid Search

POWERFUL FEATURE: Combine vector search + metadata filtering!

Example: "Find similar documents, but only from 'AI' topic"

This is HYBRID SEARCH:
- Vector similarity (semantic meaning)
- + Metadata filtering (structured data)
= Best of both worlds!

In [ ]:
query = "learning and training"
filter_dict = {"topic": "AI"}
print(query)
print(filter_dict)

# search with filter
filtered_results = vectorstore.similarity_search(
    query = query,
    k = 5,
    filter = filter_dict # filters with topic "AI"
)

print(f"Found {len(filtered_results)} documents")

learning and training
{'topic': 'AI'}
Found 1 documents


In [ ]:
for i, doc in enumerate(filtered_results, 1):
  print(f"\n{i}. Topic: {doc.metadata['topic']}")
  print(f"Content: {doc.page_content}")


1. Topic: AI
Content: Neural networks use backpropagation for training


USE CASES FOR FILTERING:
- Filter by date range (recent documents only)
- Filter by document type (PDFs only, not images)
- Filter by department (HR docs vs Engineering docs)
- Filter by language (English only)
- Combine multiple filters!

##Creating a Retriever ( RAG Component )

A RETRIEVER is the key component for RAG:
- Takes a query
- Returns relevant documents
- Configurable (top-k, filters, etc.)

This is what connects your knowledge base to your LLM!

In [ ]:
# creating a retriever
retriever = vectorstore.as_retriever(
    search_type = "similarity", # or mmr for diversity
    search_kwargs = {"k": 2}
)
print(type(retriever), "retriever type")

<class 'langchain_core.vectorstores.base.VectorStoreRetriever'> retriever type


In [ ]:
# use the retriever
query = "scientific processes in nature"
print(query)

# retrieving documents
retrieved = retriever.invoke(query)

print(f"Retrieved {len(retrieved)} documents")

scientific processes in nature
Retrieved 2 documents


In [ ]:
# retrieved documents
for i, doc in enumerate(retrieved, 1):
  print(f"\n{i}. Topic: {doc.metadata['topic']}")
  print(f"Content: {doc.page_content}")
  print(f"Metadata: {doc.metadata}")


1. Topic: biology
Content: The mitochondria is the powerhouse of the cell
Metadata: {'topic': 'biology', 'difficulty': 'medium'}

2. Topic: physics
Content: Water boils at 100 degrees Celsius at sea level
Metadata: {'topic': 'physics', 'difficulty': 'easy'}


WHY USE A RETRIEVER?
- Standard interface for RAG pipelines
- Easy to swap different retrieval strategies
- Integrates seamlessly with LLMs
- Can chain multiple retrievers

## Different Retrieval Strategies

1. SIMILARITY SEARCH (Default)
   - Returns most similar documents
   - Simple and effective
   - May return very similar/redundant results
   
2. MMR (Maximal Marginal Relevance)
   - Balances similarity + diversity
   - Avoids redundant results
   - Good when you want varied perspectives
   
3. SIMILARITY SCORE THRESHOLD
   - Only return docs above certain similarity
   - Good for quality control
   - May return 0 results if threshold too high

Let's Compare!

In [ ]:
# creating sample docs with some redundancy
redundant_docs = [
    Document(
        page_content = "Dogs are loyal pets and great companions", metadata={"type":"A"}
    ),
    Document(
        page_content = "Dogs are faithful animals and woderful friends", metadata={"type":"A"}
    ),
    Document(
        page_content = "Cats are independent pets", metadata={"type":"B"}
    ),
    Document(
        page_content = "Birds can fly in the sky", metadata={"type":"C"}
    )
]

vectorstore2 = Chroma.from_documents(
    documents = redundant_docs,
    embedding = lc_embeddings,
    collection_name = "strategy_demo",
    client = client
)

query = "faithful animal companions"
print("Query:", query)

Query: faithful animal companions


In [ ]:
#strategy 1 : similarity
retriever_sim = vectorstore2.as_retriever(
    search_type = "similarity",
    search_kwargs = {"k": 2}
)
results_sim = retriever_sim.invoke(query)
for i, doc in enumerate(results_sim, 1):
  print(f"{i}. {doc.page_content}")


1. Dogs are faithful animals and woderful friends
2. Dogs are loyal pets and great companions


In [ ]:
# strategy 2: MMR
retriever_mmr = vectorstore2.as_retriever(
    search_type = "mmr",
    search_kwargs = {"k": 2, "fetch_k": 4} # fetch 4 return diverse 2
)

results_mmr = retriever_mmr.invoke(query)
for i, doc in enumerate(results_mmr, 1):
  print(f"{i}. {doc.page_content}")

1. Dogs are faithful animals and woderful friends
2. Birds can fly in the sky


Notice the difference -

- Similarity: Might return two very similar dog docs
- MMR: Returns one dog doc + one different topic ( or same but more diverse )

# Building a Complete Retrieval System

In [ ]:
class SimpleRAGRetriever:
  """ Complete retrieval system for RAG"""
  def __init__(self, documents: List[Document], embedding_model_name: str = "all-MiniLM-L6-v2"):
    print("Documents:", len(documents))
    print("Embedding model :", embedding_model_name)

    # create embeddings
    self.embeddings = HuggingFaceEmbeddings(model_name = embedding_model_name)

    # create vector store
    self.vectorstore = Chroma.from_documents(
        documents = documents,
        embedding = self.embeddings,
        client = client,
        collection_name = "rag_retriever"
    )

  def retrieve(self, query: str, k: int= 3, use_mmr: bool = False) -> List[Document]:
    search_type = "mmr" if use_mmr else "similarity"
    retriever = self.vectorstore.as_retriever(
        search_type = search_type,
        search_kwargs= {"k": k}
    )
    results = retriever.invoke(query)
    return results

  def retrive_with_score(self, query:str, k:int=3):
    results = self.vectorstore.similarity_search_with_score(
        query = query,
        k = k
    )


In [3]:
# testing the system
test_docs = [
    Document(page_content="Retrieval-Augmented Generation combines search with LLMs",
             metadata={"source": "rag_intro.pdf", "page": 1}),
    Document(page_content="Vector databases store embeddings for fast similarity search",
             metadata={"source": "vectordb_guide.pdf", "page": 5}),
    Document(page_content="ChromaDB is an open-source embedding database",
             metadata={"source": "chromadb_docs.pdf", "page": 2}),
    Document(page_content="The Eiffel Tower was built in 1889",
             metadata={"source": "paris_facts.pdf", "page": 12}),]

rag_retriever = SimpleRAGRetriever(test_docs)
query = "how does RAG work with embeddings?"
results = rag_retriever.retrieve(query, k = 2)
print("query:", query)

for i, doc in enumerate(results, 1):
  print(f"{i}. {doc.page_content}")
  print(f"Source: {doc.metadata['source']}")
  print(f"Page: {doc.metadata['page']}")

# Performance and Indexing

CHROMADB INDEXING ( behind the scenes )-

1. HNSW (Hierarchical Navigable Small World)
   - Default in ChromaDB
   - Creates a graph structure
   - Very fast for similarity search
   - Trade-off: Uses more memory
   
2. How it works:
   - Builds a multi-layer graph of vectors
   - Searches by "navigating" the graph
   - Checks only ~1-5% of total vectors
   - Result: 100x faster than brute force!

SCALE BENCHMARKS (approximate):

10,000 vectors:     <10ms per query

100,000 vectors:    <20ms per query  

1,000,000 vectors:  <50ms per query

10,000,000 vectors: <200ms per query

FOR PRODUCTION:
- ChromaDB: Good for <1M vectors
- Pinecone/Weaviate: Better for >1M vectors
- FAISS: Best for raw speed, requires more setup

# Key Takeaways from module

1. WHY VECTOR DATABASES?
   
   ✅ Fast search through millions of embeddings
   
   ✅ Organized indexing (HNSW, IVF, etc.)
   
   ✅ Metadata filtering for hybrid search

2. CHROMADB BASICS:
   
   ✅ Collections = Tables
   
   ✅ Documents + Embeddings + Metadata
   
   ✅ L2 distance by default (lower = more similar)

3. LANGCHAIN INTEGRATION:
   
   ✅ Automatic embedding handling
   
   ✅ Clean Document interface
   
   ✅ Easy retriever creation

4. RETRIEVAL STRATEGIES:
   
   ✅ Similarity: Most similar results
   
   ✅ MMR: Balanced similarity + diversity
   
   ✅ Score threshold: Quality control

5. HYBRID SEARCH:
   
   ✅ Vector similarity (semantic)
   
   ✅ + Metadata filtering (structured)
   
   ✅ = Powerful, precise retrieval

6. THE RETRIEVER:
   ✅ Core component for RAG
   ✅ Bridges knowledge base → LLM
   ✅ Configurable and chainable
